In [ ]:
!pip install datasets transformers sentence-transformers faiss-cpu accelerate -q

In [ ]:
from datasets import load_dataset

# Usa a versão parquet do wikipedia PT, sem script
dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pt",
    split="train",
    streaming=True,
    trust_remote_code=False
)

# Pega os primeiros 500 artigos
articles = []
for i, item in enumerate(dataset):
    articles.append({"title": item["title"], "text": item["text"]})
    if i >= 499:
        break

print(f"Total de artigos carregados: {len(articles)}")
print("Exemplo:", articles[0]["title"])

In [ ]:
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # overlap aqui
    return chunks

# Gera chunks de todos os artigos
all_chunks = []
for article in articles:
    chunks = chunk_text(article["text"], chunk_size=256, overlap=30)
    for chunk in chunks:
        all_chunks.append({"title": article["title"], "text": chunk})

print(f"Total de chunks gerados: {len(all_chunks)}")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Carrega modelo de embeddings open-source
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Gera embeddings para todos os chunks
print("Gerando embeddings... (pode demorar alguns minutos)")
texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

# Cria índice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Índice FAISS criado com {index.ntotal} vetores")

In [ ]:
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])
    return results

# Teste rápido
query = "Quem foi Dom Pedro II?"
chunks_recuperados = retrieve(query)
for c in chunks_recuperados:
    print(f"\n📄 Título: {c['title']}")
    print(c["text"][:300])

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# Carrega modelo e tokenizer diretamente
model_name = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()

def generate_answer(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def answer_with_rag(query):
    chunks = retrieve(query, top_k=3)
    context = " ".join([c["text"] for c in chunks])[:800]
    prompt = f"Contexto: {context}\n\nPergunta: {query}\n\nResposta:"
    return generate_answer(prompt)

def answer_without_rag(query):
    return generate_answer(query)

# Teste de comparação
query = "Quem foi Dom Pedro II?"
print("=== COM RAG ===")
print(answer_with_rag(query))
print("\n=== SEM RAG (Closed-book) ===")
print(answer_without_rag(query))